# 03 · Weekly feature store: gate check and signal with confidence intervals (Phase 3)

**What this notebook does.** The weekly feature store (`python -m features weekly`, code in `src/features/weekly.py`)
turns 18 months of transactions into one row per account per week since opening, using only transactions up to the end
of that week. This notebook is the Phase 3 gate record:

1. **The store itself**: size, manifest, and plain consistency checks.
2. **Gate: reproduce the Phase 1 screen.** The 16 features the EDA already screened must give the same AUCs from the
   store as in `outputs/modelling/eda/feature_screen.csv` (months 0–4, weeks 4 / 8 / 12 / 26).
3. **Signal, with uncertainty.** For every feature: how well it separates a launderer's *first laundering week* from
   normal account-weeks, as an AUC with a 95% confidence interval. The interval comes from a bootstrap that resamples
   whole **accounts** (not single weeks), because one account's weeks are not independent of each other.

Only training months 0–4 are used. Month 5 is untouched, test months 6–7 stay closed. Labels come from
`amldata.labels()` only. Correctness of each feature (an independent pandas re-implementation), no-peeking on real data
and byte-identical rebuilds are tested in `tests/test_feature_store.py`.

In [1]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "amldata").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import amldata
from features import manifest, weekly

OUT = os.environ.get("EDA_DATA", amldata.DEFAULT_OUT)
NAME = os.path.basename(os.path.normpath(OUT))
REPORT = Path(os.environ.get("EDA_REPORT", "outputs/modelling/features" + ("" if NAME == "v1_full" else f"/{NAME}")))
REPORT.mkdir(parents=True, exist_ok=True)
EDA_SCREEN = Path("outputs/modelling/eda" + ("" if NAME == "v1_full" else "/v1_50k" if NAME == "v1" else f"/{NAME}")) / "feature_screen.csv"
TRAIN = (0, 1, 2, 3, 4)
HORIZON = 26
B = 1000                     # bootstrap replicates
SEED = 20261015
T0 = time.time()
lines = []
def log(s=""):
    print(s); lines.append(str(s))
log(f"data {OUT} -> report {REPORT}; EDA screen {EDA_SCREEN}")

data data/v1_full -> report outputs/modelling/features; EDA screen outputs/modelling/eda/feature_screen.csv


## 1. The store

One row per account and week since opening (weeks 0–78), 28 features plus the two keys. The manifest records the code
commit, the feature list, the cut-off rule, a fingerprint of the data run and of the frozen peer medians;
`weekly.load()` refuses a store that does not match.

In [2]:
m = manifest.check(OUT, weekly.FEATURES)
con = amldata.public(OUT)
con.execute(f"PRAGMA memory_limit='{os.environ.get('EDA_DUCKDB_MEMORY', '6GB')}'")
store_path = Path(OUT) / "features" / "weekly.parquet"
con.execute(f"CREATE VIEW s AS SELECT * FROM read_parquet('{store_path.as_posix()}')")
n_acc = con.execute("SELECT count(*) FROM accounts").fetchone()[0]
n_rows = con.execute("SELECT count(*) FROM s").fetchone()[0]
log(f"## 1. Store\n\n{n_rows:,} account-weeks = {n_acc:,} accounts x {m['weeks'][1] - m['weeks'][0] + 1} weeks; "
    f"{len(m['features'])} features; code {m['git_commit'][:7]}{' (uncommitted changes)' if m['git_uncommitted_changes'] else ''}; "
    f"cut-off rule: {m['cutoff_rule']}")
assert n_rows == n_acc * (m["weeks"][1] - m["weeks"][0] + 1) == m["rows"]
nulls = con.execute("SELECT " + ", ".join(f"avg(({c} IS NULL)::INT) AS {c}" for c in weekly.FEATURES) + " FROM s").df().T
nulls.columns = ["share_null"]
log("\nShare of account-weeks where a feature is undefined (e.g. no money in that week, fewer than 4 earlier weeks):\n")
log(nulls[nulls.share_null > 0].round(4).to_markdown())

## 1. Store

19,276,000 account-weeks = 244,000 accounts x 79 weeks; 28 features; code d52d1e0 (uncommitted changes); cut-off rule: rows with day <= open_day + 7*w + 6 (end of each account's week w, as amldata.labels)



Share of account-weeks where a feature is undefined (e.g. no money in that week, fewer than 4 earlier weeks):



|                       |   share_null |
|:----------------------|-------------:|
| new_payer_share_wk    |       0.4643 |
| passthrough_wk        |       0.4542 |
| out_1h_share_wk       |       0.4542 |
| out_24h_share_wk      |       0.4542 |
| med_in_out_minutes_wk |       0.0126 |
| self_vs_past_money_in |       0.6547 |
| self_vs_past_payers   |       0.7255 |
| self_vs_past_rows     |       0.0577 |
| peer_ratio_money_in   |       0.0163 |
| cash_in_share_cum     |       0.0103 |


## 2. Gate: the store reproduces the Phase 1 screen, value by value

The EDA (`01_eda_train.ipynb`, section 3) built its account-week panel with one SQL query; that query is copied here
**verbatim** and rerun on the same data. Two checks:

- **(A) The copy is faithful:** its AUCs equal the published `feature_screen.csv`: exactly for count features, and
  within 1e-5 for the eight features built from float money sums. DuckDB adds floats in a different order from run to
  run, so the EDA's own query does not give bit-identical money sums twice; the published screen carries this noise.
- **(B) The store implements the same definitions:** for each of the 16 features the EDA screened, every account-week
  value in the store equals the EDA panel's value (relative difference below 1e-9; undefined in both or neither).

Why value by value and not AUC alone: the EDA adds money as binary floats, the store as exact pence. Values then differ
by about 1e-12 of a pound, but some features have many exact ties (e.g. 40% of self-vs-past values at week 4), and the
EDA's rounding noise splits those ties at random, which moves its AUC by up to ~2e-3. The store keeps true ties.

In [3]:
MAX_WEEK = HORIZON
HIGH_RISK = weekly.HIGH_RISK
months_sql = ", ".join(map(str, TRAIN))
con.execute(f"PRAGMA temp_directory='{(Path(OUT) / 'features' / 'duckdb_tmp').as_posix()}'")
# ---- verbatim copy of PANEL from notebooks/01_eda_train.ipynb, section 3 ----
PANEL = f"""
WITH acc AS (
    SELECT account_id, open_day, income, customer_age FROM accounts WHERE open_month IN ({months_sql})),
t AS (
    SELECT x.account_id, x.direction, x.counterparty_id, x.amount, x.payment_type, x.receiver_bank_location,
           (x.day - a.open_day) // 7 AS w,
           x.day * 86400 + CAST(substr(x.time, 1, 2) AS INT) * 3600 + CAST(substr(x.time, 4, 2) AS INT) * 60
               + CAST(substr(x.time, 7, 2) AS INT) AS ts,
           CAST(substr(x.time, 1, 2) AS INT) < 6 AS night,
           row_number() OVER (PARTITION BY x.account_id, x.counterparty_id ORDER BY x.day, x.time) AS nth_with_cp
    FROM transactions x JOIN acc a USING (account_id)
    WHERE x.day <= a.open_day + 7 * {MAX_WEEK} + 6),
ins AS (SELECT account_id, ts FROM t WHERE direction = 'in'),
outs AS (
    SELECT o.account_id, o.w, o.amount, o.ts - i.ts AS delay
    FROM (SELECT account_id, w, amount, ts FROM t WHERE direction = 'out') o
    ASOF LEFT JOIN ins i ON o.account_id = i.account_id AND o.ts >= i.ts),
speed AS (
    SELECT account_id, w,
           coalesce(sum(amount) FILTER (WHERE delay <= 3600), 0)  AS out_1h,
           coalesce(sum(amount) FILTER (WHERE delay <= 86400), 0) AS out_24h,
           median(delay) / 60.0 AS med_delay_min
    FROM outs GROUP BY 1, 2),
wk AS (
    SELECT account_id, w,
           count(*) AS n_rows,
           coalesce(sum(amount) FILTER (WHERE direction = 'in'), 0)  AS money_in,
           coalesce(sum(amount) FILTER (WHERE direction = 'out'), 0) AS money_out,
           coalesce(max(amount) FILTER (WHERE direction = 'in'), 0)  AS max_in,
           count(DISTINCT counterparty_id) FILTER (WHERE direction = 'in' AND counterparty_id <> 'CASH') AS n_payers,
           count(DISTINCT counterparty_id) FILTER (WHERE direction = 'out' AND counterparty_id NOT IN ('CASH', 'CARD_MERCHANTS')) AS n_payees,
           count(DISTINCT counterparty_id) FILTER (WHERE direction = 'in' AND counterparty_id <> 'CASH' AND nth_with_cp = 1) AS n_new_payers,
           avg(CASE WHEN payment_type LIKE 'Cash%' THEN 1 ELSE 0 END) AS cash_share,
           coalesce(sum(amount) FILTER (WHERE direction = 'out' AND counterparty_id LIKE 'A%'), 0) AS internal_out,
           coalesce(sum(amount) FILTER (WHERE direction = 'out' AND receiver_bank_location <> 'UK'), 0) AS abroad_out,
           coalesce(sum(amount) FILTER (WHERE direction = 'out' AND receiver_bank_location IN ('{"', '".join(HIGH_RISK)}')), 0) AS hr_out,
           avg(CASE WHEN night THEN 1 ELSE 0 END) AS night_share
    FROM t GROUP BY 1, 2),
grid AS (SELECT account_id, open_day, income, customer_age, unnest(range(0, {MAX_WEEK + 1})) AS w FROM acc),
g AS (
    SELECT grid.account_id, grid.w, grid.income, grid.customer_age,
           coalesce(n_rows, 0) n_rows, coalesce(money_in, 0) money_in, coalesce(money_out, 0) money_out,
           coalesce(max_in, 0) max_in, coalesce(n_payers, 0) n_payers, coalesce(n_payees, 0) n_payees,
           coalesce(n_new_payers, 0) n_new_payers, cash_share, coalesce(internal_out, 0) internal_out,
           coalesce(abroad_out, 0) abroad_out, coalesce(hr_out, 0) hr_out, night_share,
           coalesce(out_1h, 0) out_1h, coalesce(out_24h, 0) out_24h, med_delay_min
    FROM grid LEFT JOIN wk USING (account_id, w) LEFT JOIN speed USING (account_id, w))
SELECT *,
    n_new_payers / nullif(n_payers, 0)                                   AS new_payer_share,
    least(money_out, money_in) / nullif(money_in, 0)                     AS passthrough_wk,
    out_1h / nullif(money_in, 0)                                         AS out_1h_share,
    out_24h / nullif(money_in, 0)                                        AS out_24h_share,
    internal_out / nullif(money_out, 0)                                  AS internal_share,
    abroad_out / nullif(money_out, 0)                                    AS abroad_share,
    hr_out / nullif(money_out, 0)                                        AS hr_share,
    sum(money_in) OVER (PARTITION BY account_id ORDER BY w)              AS money_in_cum,
    sum(n_rows)   OVER (PARTITION BY account_id ORDER BY w)              AS n_rows_cum,
    (money_in - median(money_in) OVER past) / nullif(mad(money_in) OVER past, 0) AS self_vs_past_in,
    (n_payers - median(n_payers) OVER past) / nullif(mad(n_payers) OVER past, 0) AS self_vs_past_payers,
    count(*) OVER past                                                   AS n_past_weeks,
    money_in / nullif(median(money_in) OVER (PARTITION BY income, customer_age, w), 0) AS peer_ratio_in,
    money_in / nullif(median(money_in) OVER (PARTITION BY income, w), 0)               AS income_fit_in
FROM g
WINDOW past AS (PARTITION BY account_id ORDER BY w ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
"""
# ---- end of copy ----
con.execute(f"CREATE OR REPLACE TABLE panel AS {PANEL}")
con.execute("UPDATE panel SET self_vs_past_in = NULL, self_vs_past_payers = NULL WHERE n_past_weeks < 4")
panel = con.execute("SELECT * FROM panel").df()

In [4]:
def auc(x, y):
    """Rank AUC (Mann-Whitney); NaN ranked lowest, as in the EDA. 0.5 = no signal."""
    y = np.asarray(y).astype(bool)
    n_pos, n_neg = y.sum(), (~y).sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    # na_option="top" gives NaN the lowest ranks ("bottom" would rank it above every value)
    r = pd.Series(np.asarray(x, dtype=float)).rank(method="average", na_option="top").to_numpy()
    return (r[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

SAME = {"n_rows_wk": "n_rows", "money_in_wk": "money_in", "money_out_wk": "money_out", "max_in_wk": "max_in",
        "n_payers_wk": "n_payers", "n_payees_wk": "n_payees", "new_payer_share_wk": "new_payer_share",
        "passthrough_wk": "passthrough_wk", "out_1h_share_wk": "out_1h_share", "out_24h_share_wk": "out_24h_share",
        "med_in_out_minutes_wk": "med_delay_min", "money_in_cum": "money_in_cum",
        "n_rows_cum": "n_rows_cum", "self_vs_past_money_in": "self_vs_past_in",
        "self_vs_past_payers": "self_vs_past_payers", "peer_ratio_money_in": "peer_ratio_in"}
store26 = weekly.load(OUT, (0, HORIZON))
train_ids = set(amldata.labels(OUT, months=TRAIN)["account_id"])
store26 = store26[store26["account_id"].isin(train_ids)]
eda_panel = panel.rename(columns=lambda c: {"w": "weeks_since_open", "account_id": "account_id"}.get(c, "eda_" + c))
both = store26.merge(eda_panel, on=["account_id", "weeks_since_open"], how="inner")
assert len(both) == len(store26) == len(panel)

# (B) value by value
val = []
for mine, theirs in SAME.items():
    a, b = both[mine].to_numpy(float), both["eda_" + theirs].to_numpy(float)
    same_null = np.isnan(a) == np.isnan(b)
    rel = np.abs(a - b) / np.maximum(1.0, np.abs(b))
    val.append({"feature": mine, "eda_name": theirs, "account_weeks": len(a), "null_pattern_equal": bool(same_null.all()),
                "max_rel_diff": float(np.nanmax(np.where(np.isnan(a) | np.isnan(b), 0, rel)))})
val = pd.DataFrame(val)

# (A) the copy reproduces the published screen; and the store's AUCs beside it
eda = pd.read_csv(EDA_SCREEN)
rows = []
for W in (4, 8, 12, 26):
    lw = amldata.labels(OUT, months=TRAIN, weeks_since_open=W)
    pw = both[both.weeks_since_open == W].merge(lw, on="account_id")
    pw = pw[pw.open_by_day == 1]                 # accounts the bank has not closed, as in the EDA
    for mine, theirs in SAME.items():
        e = eda[(eda.week == W) & (eda.feature == theirs)].iloc[0]
        for lab, col in [("mule_by_day", "auc_mule_by_day")]:
            if pd.isna(e[col]):
                continue
            rows.append({"week": W, "feature": mine, "eda_published": e[col],
                         "eda_query_rerun": auc(pw["eda_" + theirs], pw[lab]), "store": auc(pw[mine], pw[lab])})
au = pd.DataFrame(rows)
au["rerun_vs_published"] = (au.eda_query_rerun - au.eda_published).abs()
au["store_vs_published"] = (au.store - au.eda_published).abs()
FLOAT_SUMS = {"money_in_wk", "money_out_wk", "passthrough_wk", "money_in_cum", "self_vs_past_money_in", "peer_ratio_money_in",
              "out_1h_share_wk", "out_24h_share_wk"}
au["float_sum_feature"] = au.feature.isin(FLOAT_SUMS)
au.to_csv(REPORT / "gate_auc.csv", index=False)
val.to_csv(REPORT / "gate_values.csv", index=False)

log("\n## 2. Gate: Phase 1 screen reproduced\n")
log(f"(A) EDA query rerun vs published screen ({len(au)} AUCs, mule_by_day): max |difference| "
    f"{au.loc[~au.float_sum_feature, 'rerun_vs_published'].max():.1e} for count features, "
    f"{au.loc[au.float_sum_feature, 'rerun_vs_published'].max():.1e} for the eight float-sum money features")
log(f"(B) store vs EDA panel, value by value over {len(both):,} account-weeks (months 0-4, weeks 0-{HORIZON}):\n")
log(val.to_markdown(index=False, floatfmt=".1e"))
log(f"\nStore AUC vs published AUC: max |difference| {au.store_vs_published.max():.1e} "
    f"(largest for {au.loc[au.store_vs_published.idxmax(), 'feature']}, week {au.loc[au.store_vs_published.idxmax(), 'week']}); "
    "differences come only from float ties, since (B) shows the values agree.")
assert au.loc[~au.float_sum_feature, "rerun_vs_published"].max() < 1e-12, "(A) failed on count features"
assert au.loc[au.float_sum_feature, "rerun_vs_published"].max() < 1e-5, "(A) failed beyond float-sum noise"
assert val.null_pattern_equal.all() and val.max_rel_diff.max() < 1e-9, "(B) failed: store values differ from the EDA panel"
log(f"\nGATE PASSED: (A) and (B) hold for all {len(SAME)} shared features.")


## 2. Gate: Phase 1 screen reproduced

(A) EDA query rerun vs published screen (64 AUCs, mule_by_day): max |difference| 2.2e-16 for count features, 1.3e-06 for the eight float-sum money features
(B) store vs EDA panel, value by value over 4,450,113 account-weeks (months 0-4, weeks 0-26):

| feature               | eda_name            |   account_weeks | null_pattern_equal   |   max_rel_diff |
|:----------------------|:--------------------|----------------:|:---------------------|---------------:|
| n_rows_wk             | n_rows              |         4450113 | True                 |        0.0e+00 |
| money_in_wk           | money_in            |         4450113 | True                 |        4.1e-16 |
| money_out_wk          | money_out           |         4450113 | True                 |        5.4e-16 |
| max_in_wk             | max_in              |         4450113 | True                 |        0.0e+00 |
| n_payers_wk           | n_payers            |         4450113 | True   

In [5]:
# the full Phase 3 screen of all 28 features from the store (same protocol as the EDA), for the record
rows = []
for W in (4, 8, 12, 26):
    lw = amldata.labels(OUT, months=TRAIN, weeks_since_open=W)
    pw = store26[store26.weeks_since_open == W].merge(lw, on="account_id")
    pw = pw[pw.open_by_day == 1]                 # accounts the bank has not closed, as in the EDA
    started = pw.started_by_day.to_numpy() == 1
    nrm = pw.is_launderer.to_numpy() == 0
    for c in weekly.FEATURES:
        x = pw[c].to_numpy(dtype=float)
        rows.append({"week": W, "feature": c,
                     "auc_mule_by_day": auc(x, pw.mule_by_day) if pw.mule_by_day.sum() >= 20 else np.nan,
                     "auc_started_by_day": auc(x[started | nrm], started[started | nrm]) if started.sum() >= 20 else np.nan})
pd.DataFrame(rows).to_csv(REPORT / "feature_screen.csv", index=False)

## 3. Signal in the first laundering week, with 95% confidence intervals

**The question.** Activity can only show laundering once it has started. So for each training launderer we take its
*first laundering week* (the first week where `started_by_day` is 1) and compare it with every open account-week of a
normal account, weeks 0–26. AUC = the chance that a random launderer's first week scores higher than a random normal
week (0.5 = no signal; below 0.5 = launderers score *lower*, e.g. shorter delays).

**The interval.** Weeks of one account are alike, so treating 4 million normal weeks as independent would make the
interval far too narrow. We resample whole accounts instead (Poisson bootstrap, 1,000 replicates: each account gets a
random weight, all its weeks share it) and take the 2.5% and 97.5% points. To keep this fast, normal accounts are a
seeded random 20,000 of the training accounts (about 0.5 million account-weeks); every training launderer is kept.

In [6]:
labs = []
for w in range(HORIZON + 1):
    l = amldata.labels(OUT, months=TRAIN, weeks_since_open=w)[["account_id", "is_launderer", "started_by_day", "open_by_day"]]
    l["weeks_since_open"] = w
    labs.append(l)
d = store26.merge(pd.concat(labs), on=["account_id", "weeks_since_open"])
d = d[d.open_by_day == 1]
first = d[d.started_by_day == 1].groupby("account_id").weeks_since_open.min()
pos = d[d.account_id.isin(first.index) & (d.weeks_since_open == d.account_id.map(first))]
rng = np.random.default_rng(SEED)
normal_ids = d.loc[d.is_launderer == 0, "account_id"].unique()
keep_normal = rng.choice(normal_ids, size=min(20_000, len(normal_ids)), replace=False)
neg = d[d.account_id.isin(keep_normal) & (d.is_launderer == 0)]
log(f"\n## 3. First-laundering-week signal\n\n{len(pos)} launderers start inside weeks 0-{HORIZON} "
    f"(of {d[d.is_launderer == 1].account_id.nunique()} training launderers); {len(neg):,} normal account-weeks "
    f"from {neg.account_id.nunique():,} normal accounts.")

y = np.r_[np.ones(len(pos), bool), np.zeros(len(neg), bool)]
acct = pd.Index(np.r_[pos.account_id.to_numpy(), neg.account_id.to_numpy()])
codes, uniq = pd.factorize(acct)
W = rng.poisson(1.0, size=(B, len(uniq))).astype(float)      # one weight per account per replicate

def groups(score):
    """Rank group of each row (equal scores share a group; NaN lowest, as in the EDA). Sorted once per feature."""
    return np.unique(np.where(np.isnan(score), -np.inf, score), return_inverse=True)[1]

def weighted_auc(grp, y, w):
    """AUC with row weights; ties count half."""
    wp = np.bincount(grp, weights=w * y)
    wn = np.bincount(grp, weights=w * ~y)
    below = np.cumsum(wn) - wn
    return float((wp * (below + 0.5 * wn)).sum() / (wp.sum() * wn.sum()))

res = []
for c in weekly.FEATURES:
    x = np.r_[pos[c].to_numpy(float), neg[c].to_numpy(float)]
    g = groups(x)
    point = weighted_auc(g, y, np.ones(len(y)))
    boots = np.array([weighted_auc(g, y, W[b, codes]) for b in range(B)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    res.append({"feature": c, "auc": point, "ci_low": lo, "ci_high": hi,
                "separation": abs(point - 0.5), "excludes_0.5": (lo > 0.5) or (hi < 0.5)})
sig = pd.DataFrame(res).sort_values("separation", ascending=False).reset_index(drop=True)
sig.to_csv(REPORT / "first_week_auc_ci.csv", index=False)
log("\nAUC of each feature, first laundering week vs normal account-weeks, 95% account-level bootstrap interval:\n")
log(sig.drop(columns="separation").to_markdown(index=False, floatfmt=".3f"))
log(f"\n{int(sig['excludes_0.5'].sum())} of {len(sig)} features have an interval that excludes 0.5 (no signal).")


## 3. First-laundering-week signal

361 launderers start inside weeks 0-26 (of 455 training launderers); 540,000 normal account-weeks from 20,000 normal accounts.



AUC of each feature, first laundering week vs normal account-weeks, 95% account-level bootstrap interval:

| feature                 |   auc |   ci_low |   ci_high | excludes_0.5   |
|:------------------------|------:|---------:|----------:|:---------------|
| out_1h_share_wk         | 0.954 |    0.944 |     0.964 | True           |
| n_in_wk                 | 0.915 |    0.904 |     0.927 | True           |
| out_24h_share_wk        | 0.907 |    0.900 |     0.914 | True           |
| n_rows_wk               | 0.906 |    0.889 |     0.922 | True           |
| money_out_wk            | 0.903 |    0.885 |     0.921 | True           |
| self_vs_past_rows       | 0.877 |    0.853 |     0.898 | True           |
| n_payers_wk             | 0.853 |    0.835 |     0.872 | True           |
| max_out_wk              | 0.848 |    0.828 |     0.868 | True           |
| money_in_wk             | 0.839 |    0.829 |     0.849 | True           |
| peer_ratio_money_in     | 0.825 |    0.812 |     0.837

### Speed: why the store keeps the EDA's definition (decision 15 Oct 2026)

The store's `out_1h_share_wk` / `out_24h_share_wk` are the EDA's definitions: money paid out (any kind) within 1 h /
24 h of the latest payment in (any kind), over the week's money in. The Phase 1 report had suggested a redefinition,
because the 24 h share is noisy on normal accounts: **transfers** out within 1 h / 24 h of the latest **transfer** in,
each inflow passed on at most once. (Correction 18 Oct 2026: the Phase 1 screen that made the speed shares look swamped
ranked missing values highest by mistake; ranked lowest, they separate started launderers well.) That version was built
and tested on 15 Oct and dropped. This cell rebuilds it inline so the evidence stays reproducible, and compares both on
the same first-laundering weeks and normal weeks with a **paired** bootstrap (both scored on the same resampled accounts), the right test for two features measured on the
same data. A breakdown by laundering pattern needs the hidden pattern labels, so it lives in
`src/evaluation/speed_by_pattern.py` (output `speed_by_pattern.csv` next to this report).

In [7]:
tried = con.execute(f'''
    WITH acc AS (SELECT account_id, open_day FROM accounts WHERE open_month IN ({", ".join(map(str, TRAIN))})),
    t AS (SELECT x.account_id, x.direction, x.payment_type, CAST(x.amount AS DECIMAL(18, 2)) AS amount,
                 (x.day - a.open_day) // 7 AS w,
                 x.day * 86400 + CAST(substr(x.time, 1, 2) AS INT) * 3600 + CAST(substr(x.time, 4, 2) AS INT) * 60
                     + CAST(substr(x.time, 7, 2) AS INT) AS ts
          FROM transactions x JOIN acc a USING (account_id) WHERE x.day <= a.open_day + 7 * {HORIZON} + 6),
    tin AS (SELECT account_id, ts, amount AS in_amt, row_number() OVER (PARTITION BY account_id ORDER BY ts, amount) AS in_id
            FROM t WHERE direction = 'in' AND payment_type = 'ACH'),
    fo AS (SELECT o.account_id, o.w, o.ts, o.amount, i.in_id, i.in_amt, o.ts - i.ts AS delay
           FROM (SELECT * FROM t WHERE direction = 'out' AND payment_type IN ('ACH', 'Cross-border')) o
           ASOF JOIN tin i ON o.account_id = i.account_id AND o.ts >= i.ts),
    fc AS (SELECT account_id, w,
                  CASE WHEN delay <= 3600 THEN greatest(0, least(amount, in_amt
                       - coalesce(sum(amount) FILTER (WHERE delay <= 3600) OVER bf, 0))) ELSE 0 END AS f1h,
                  CASE WHEN delay <= 86400 THEN greatest(0, least(amount, in_amt
                       - coalesce(sum(amount) FILTER (WHERE delay <= 86400) OVER bf, 0))) ELSE 0 END AS f24h
           FROM fo WINDOW bf AS (PARTITION BY account_id, in_id ORDER BY ts, amount
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)),
    fwd AS (SELECT account_id, w, sum(f1h) AS f1h, sum(f24h) AS f24h FROM fc GROUP BY 1, 2),
    tin_wk AS (SELECT account_id, w, sum(amount) AS transfer_in FROM t WHERE direction = 'in' AND payment_type = 'ACH' GROUP BY 1, 2)
    SELECT account_id, w AS weeks_since_open,
           least(1.0, coalesce(f1h, 0) / transfer_in) AS tried_1h, least(1.0, coalesce(f24h, 0) / transfer_in) AS tried_24h
    FROM tin_wk LEFT JOIN fwd USING (account_id, w) WHERE transfer_in > 0''').df()
both = pd.concat([pos.assign(y=True), neg.assign(y=False)]).merge(tried, on=["account_id", "weeks_since_open"], how="left")
codes2 = pd.factorize(pd.Index(both.account_id))[0]
W2 = rng.poisson(1.0, size=(B, codes2.max() + 1)).astype(float)
cmp_rows, boot = [], {}
for c in ["out_1h_share_wk", "tried_1h", "out_24h_share_wk", "tried_24h"]:
    x, yy = both[c].to_numpy(float), both.y.to_numpy()
    g = groups(x)
    boots = np.array([weighted_auc(g, yy, W2[b, codes2]) for b in range(B)])
    boot[c] = boots
    cmp_rows.append({"feature": c, "definition": "kept: EDA (any payment)" if c.startswith("out") else "dropped: transfers only, capped",
                     "auc": weighted_auc(g, yy, np.ones(len(yy))), "ci_low": np.percentile(boots, 2.5),
                     "ci_high": np.percentile(boots, 97.5),
                     "mean_first_week": np.nanmean(x[yy]), "mean_normal_week": np.nanmean(x[~yy])})
speed = pd.DataFrame(cmp_rows)
speed.to_csv(REPORT / "speed_definitions.csv", index=False)
log("\nSpeed shares, kept vs dropped definition (same account-weeks, 95% account-level bootstrap):\n")
log(speed.to_markdown(index=False, floatfmt=".3f"))
pt = speed.set_index("feature").auc
gain = []
for kept, dropped in [("out_1h_share_wk", "tried_1h"), ("out_24h_share_wk", "tried_24h")]:
    d_ = boot[kept] - boot[dropped]
    gain.append({"comparison": f"{kept} - {dropped}", "auc_gain": pt[kept] - pt[dropped],
                 "ci_low": np.percentile(d_, 2.5), "ci_high": np.percentile(d_, 97.5),
                 "share_of_replicates_kept_better": (d_ > 0).mean()})
gain = pd.DataFrame(gain)
log("\nPaired bootstrap of the AUC difference, kept minus dropped (same resampled accounts for both):\n")
log(gain.to_markdown(index=False, floatfmt=".3f"))
for _, r in gain.iterrows():
    verdict = "kept better" if r.ci_low > 0 else "dropped better" if r.ci_high < 0 else "no clear difference"
    log(f"- {r.comparison}: {verdict} (difference {r.auc_gain:+.3f}, 95% interval {r.ci_low:+.3f} to {r.ci_high:+.3f})")


Speed shares, kept vs dropped definition (same account-weeks, 95% account-level bootstrap):

| feature          | definition                      |   auc |   ci_low |   ci_high |   mean_first_week |   mean_normal_week |
|:-----------------|:--------------------------------|------:|---------:|----------:|------------------:|-------------------:|
| out_1h_share_wk  | kept: EDA (any payment)         | 0.954 |    0.944 |     0.964 |             0.404 |              0.031 |
| tried_1h         | dropped: transfers only, capped | 0.919 |    0.902 |     0.936 |             0.366 |              0.007 |
| out_24h_share_wk | kept: EDA (any payment)         | 0.907 |    0.900 |     0.913 |             0.731 |              0.588 |
| tried_24h        | dropped: transfers only, capped | 0.909 |    0.895 |     0.923 |             0.544 |              0.109 |

Paired bootstrap of the AUC difference, kept minus dropped (same resampled accounts for both):

| comparison                   |   auc_gain |  

## 4. Consistency checks on the whole store

Every account-week present once; counts and money never negative or missing; shares inside [0, 1] (the speed shares
can exceed 1: more can leave in a week than arrived); the 1-hour speed share
never above the 24-hour share; running totals never fall; and at week 78 each account's running totals equal its rows
and money in the transactions file (nothing lost, nothing counted twice). The same checks run in
`tests/test_feature_store.py::test_whole_store_invariants`.

In [8]:
q = lambda sql: con.execute(sql).fetchone()
shares = ["new_payer_share_wk", "passthrough_wk", "internal_out_share_cum",   # out_*_share_wk may exceed 1
          "cash_in_share_cum", "abroad_out_share_cum", "high_risk_out_share_cum"]
checks = {
    "keys unique and complete": q("SELECT count(DISTINCT (account_id, weeks_since_open)) FROM s")[0] == n_rows,
    "shares inside [0, 1]": q("SELECT " + " + ".join(f"count(*) FILTER (WHERE {c} < 0 OR {c} > 1 + 1e-12)" for c in shares) + " FROM s")[0] == 0,
    "speed shares: 0 <= 1 h <= 24 h": q("SELECT count(*) FROM s WHERE out_1h_share_wk < 0 OR out_1h_share_wk > out_24h_share_wk + 1e-12")[0] == 0,
    "running totals never fall": q('''SELECT count(*) FROM (SELECT money_in_cum - lag(money_in_cum) OVER a AS d1,
                                         n_payers_cum - lag(n_payers_cum) OVER a AS d2 FROM s
                                         WINDOW a AS (PARTITION BY account_id ORDER BY weeks_since_open))
                                      WHERE d1 < -1e-6 OR d2 < 0''')[0] == 0,
}
tot = q('''WITH t AS (SELECT account_id, count(*) n, sum(amount) FILTER (WHERE direction = 'in') i FROM transactions GROUP BY 1)
           SELECT max(abs(s.n_rows_cum - coalesce(t.n, 0))), max(abs(s.money_in_cum - coalesce(t.i, 0)))
           FROM s LEFT JOIN t USING (account_id) WHERE s.weeks_since_open = 78''')
checks["week-78 totals equal the transactions file"] = tot[0] == 0 and tot[1] < 1e-3
log("\n## 4. Consistency checks\n")
for k, v in checks.items():
    log(f"- {'PASS' if v else 'FAIL'}: {k}")
assert all(checks.values())
log(f"\ndone in {time.time() - T0:.0f}s")
(REPORT / "report.md").write_text("# Weekly feature store: Phase 3 gate\n\n" + "\n".join(lines) + "\n", encoding="utf-8")
print(f"report -> {REPORT / 'report.md'}")


## 4. Consistency checks

- PASS: keys unique and complete
- PASS: shares inside [0, 1]
- PASS: speed shares: 0 <= 1 h <= 24 h
- PASS: running totals never fall
- PASS: week-78 totals equal the transactions file

done in 139s
report -> outputs/modelling/features/report.md
